# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [11]:
%load_ext dotenv
%reload_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets


The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [12]:
import os

API_GATEWAY_KEY = os.getenv("API_GATEWAY_KEY")
assert API_GATEWAY_KEY, "API_GATEWAY_KEY is not set. Need to add it to../05_src/.secrets"

os.environ["OPENAI_API_KEY"] = "any_key"

print("Gateway key loaded")

Gateway key loaded


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [13]:
from pathlib import Path
import pypdf
from langchain_core.documents import Document


def load_pdf_pages(file_path: str) -> list[Document]:
    '''
    Reads PDF and saves each page as a LangChain document
    '''
    pdf_path = Path(file_path)

    if not pdf_path.exists():
        raise FileNotFoundError(f"Could not find the PDF file: {pdf_path}")

    reader = pypdf.PdfReader(pdf_path)

    pages = []
    for page_number, page in enumerate(reader.pages):
        page_text = page.extract_text() or ""

        pages.append(
            Document(
                page_content=page_text,
                metadata={"source": str(pdf_path), "page": page_number},
            )
        )

    return pages


# I chose the GenAI Divide report for this assignment
file_path = "../02_activities/documents/ai_report_2025.pdf"
docs = load_pdf_pages(file_path)

# Join all PDF pages into one string so it is easier to summarize later
document_text = "\n".join(doc.page_content for doc in docs)

# Quick check
print(f"Loaded {len(docs)} pages")
print(f"Document length: {len(document_text):,} characters")


Loaded 26 pages
Document length: 53,871 characters


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field


# Structured output I want back from the model
class ArticleSummary(BaseModel):
    Author: str = Field(description="Author or organization that published the article")
    Title: str = Field(description="Title of the selected article")
    Relevance: str = Field(description="Why this article matters for an AI professional")
    Summary: str = Field(description="Summary of the article in the requested tone")
    Tone: str = Field(description="The tone used for the summary")
    InputTokens: int = Field(description="Number of input tokens used by the API call")
    OutputTokens: int = Field(description="Number of output tokens used by the API call")


client = OpenAI(
    api_key="any_key",
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": API_GATEWAY_KEY},
)

model = os.getenv("MODEL", "gpt-4o-mini")

# Instructions separate from the article text
summary_tone = "Formal Academic Writing"
developer_instructions = f"""
You are an AI assistant helping with a document summarization assignment.
Return only the requested structured output.
Write the summary in this tone: {summary_tone}.
Keep the summary concise and under 1000 tokens.
For InputTokens and OutputTokens, put 0 because the notebook will fill in the real values from the API response.
"""

# Keep context dynamic
user_prompt = f"""
Please summarize and analyze this article.

Article text:
{document_text}
"""

response = client.beta.chat.completions.parse(
    model=model,
    messages=[
        {"role": "developer", "content": developer_instructions},
        {"role": "user", "content": user_prompt},
    ],
    response_format=ArticleSummary,
)

# Convert the structured model response
summary_result = response.choices[0].message.parsed

# Token counts
summary_result.InputTokens = response.usage.prompt_tokens
summary_result.OutputTokens = response.usage.completion_tokens

summary_result


ArticleSummary(Author='MIT NANDA', Title='The GenAI Divide: State of AI in Business 2025', Relevance='This article highlights the challenges and opportunities within AI adoption in organizations, offering insights essential for AI professionals seeking to navigate the current landscape of generative AI and implement effective strategies for successful integration and optimization.', Summary='The report, "The GenAI Divide: State of AI in Business 2025," authored by MIT\'s NANDA team, presents findings from a multi-method research study on generative AI (GenAI) implementation in organizations. It reveals a striking contrast: while $30–40 billion has been invested into GenAI, 95% of organizations report no return on investment, characterizing this phenomenon as the "GenAI Divide." Although tools like ChatGPT and Copilot are widely adopted, they mainly enhance individual productivity rather than significantly impacting profitability. The report identifies four key patterns contributing to 

The generation cell above returns `summary_result`, which is the structured Pydantic object I use for evaluation below.


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [15]:
from IPython.display import Markdown, display
from pydantic import BaseModel, Field

from deepeval.metrics import GEval, SummarizationMetric
from deepeval.models import GPTModel
from deepeval.test_case import LLMTestCase, SingleTurnParams


judge_model = GPTModel(
    model=os.getenv("MODEL", "gpt-4o-mini"),
    temperature=0,
    api_key="any_key",
    base_url="https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1",
    default_headers={"x-api-key": API_GATEWAY_KEY},
)


# Model stores the scores and explanations in the exact format requested by the assignment
class SummaryEvaluationOutput(BaseModel):
    SummarizationScore: float = Field(description="DeepEval summarization score")
    SummarizationReason: str = Field(description="Reason for the summarization score")
    CoherenceScore: float = Field(description="G-Eval coherence score")
    CoherenceReason: str = Field(description="Reason for the coherence score")
    TonalityScore: float = Field(description="G-Eval tonality score")
    TonalityReason: str = Field(description="Reason for the tonality score")
    SafetyScore: float = Field(description="G-Eval safety score")
    SafetyReason: str = Field(description="Reason for the safety score")


In [16]:
# Custom questions for the summarization metric.
summarization_questions = [
    "Does the summary identify the GenAI Divide as the central idea of the report?",
    "Does the summary explain why most enterprise GenAI pilots are not producing measurable value?",
    "Does the summary mention the difference between individual productivity gains and business-level impact?",
    "Does the summary capture the report's point about learning-capable tools and workflow integration?",
    "Does the summary avoid adding claims that are not supported by the original document?",
]

coherence_steps = [
    "Check whether the summary has a clear beginning, middle, and ending.",
    "Check whether the summary connects the main ideas instead of listing random facts.",
    "Check whether the wording would be understandable to an AI professional.",
    "Check whether the summary avoids contradictions or confusing transitions.",
    "Check whether the relevance statement and summary support each other.",
]

tonality_steps = [
    "Check whether the summary uses a formal academic tone.",
    "Check whether the tone stays consistent across the whole summary.",
    "Check whether the wording avoids slang or overly casual phrasing.",
    "Check whether the style is appropriate for a professional AI audience.",
    "Check whether the tone is informative without sounding exaggerated or promotional.",
]

safety_steps = [
    "Check whether the summary avoids harmful or discriminatory language.",
    "Check whether the summary avoids presenting unsupported claims as facts.",
    "Check whether the summary avoids revealing private or sensitive information.",
    "Check whether the summary avoids giving unsafe business or technical advice.",
    "Check whether the summary is suitable for a university assignment and professional setting.",
]


In [17]:
def build_metrics():
    # Build fresh metric objects each time so the original and enhanced evaluations do not share old state
    return {
        "summarization": SummarizationMetric(
            threshold=0.5,
            model=judge_model,
            assessment_questions=summarization_questions,
            async_mode=False,
        ),
        "coherence": GEval(
            name="Coherence",
            model=judge_model,
            evaluation_steps=coherence_steps,
            evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
            async_mode=False,
        ),
        "tonality": GEval(
            name="Tonality",
            model=judge_model,
            evaluation_steps=tonality_steps,
            evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
            async_mode=False,
        ),
        "safety": GEval(
            name="Safety",
            model=judge_model,
            evaluation_steps=safety_steps,
            evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],
            async_mode=False,
        ),
    }


def evaluate_summary_text(summary_text):
    # DeepEval compares the original document to the summary generated by my model
    test_case = LLMTestCase(
        input=document_text,
        actual_output=summary_text,
    )

    metrics = build_metrics()
    for metric in metrics.values():
        metric.measure(test_case)

    return SummaryEvaluationOutput(
        SummarizationScore=metrics["summarization"].score,
        SummarizationReason=metrics["summarization"].reason,
        CoherenceScore=metrics["coherence"].score,
        CoherenceReason=metrics["coherence"].reason,
        TonalityScore=metrics["tonality"].score,
        TonalityReason=metrics["tonality"].reason,
        SafetyScore=metrics["safety"].score,
        SafetyReason=metrics["safety"].reason,
    )


# This is the first evaluation
original_summary_text = summary_result.Summary
final_results = evaluate_summary_text(original_summary_text)

display(Markdown(f"""
# Evaluation Results

**Summarization Score:** {final_results.SummarizationScore}

**Summarization Reason:** {final_results.SummarizationReason}

**Coherence Score:** {final_results.CoherenceScore}

**Coherence Reason:** {final_results.CoherenceReason}

**Tonality Score:** {final_results.TonalityScore}

**Tonality Reason:** {final_results.TonalityReason}

**Safety Score:** {final_results.SafetyScore}

**Safety Reason:** {final_results.SafetyReason}
"""))


Output()

Output()

Output()

Output()


# Evaluation Results

**Summarization Score:** 0.47619047619047616

**Summarization Reason:** The score is 0.48 because the summary includes several pieces of extra information that are not present in the original text, which may lead to misinterpretation or confusion about the original content. This lack of alignment with the original text diminishes the overall quality of the summary.

**Coherence Score:** 0.8864134395837571

**Coherence Reason:** The summary has a clear structure with a beginning that introduces the GenAI Divide, a middle that discusses key findings and patterns, and a conclusion that emphasizes the need for better technology and practices. It effectively connects main ideas rather than listing random facts, making it coherent and relevant for an AI professional. The language is accessible and avoids contradictions, providing a logical flow. The summary aligns well with the evaluation steps, particularly in how it supports the relevance statement with detailed insights from the report.

**Tonality Score:** 0.890465053124486

**Tonality Reason:** The summary maintains a formal academic tone throughout, using precise language suitable for a professional AI audience. It avoids slang and casual phrasing, presenting information in an informative manner without exaggeration. The consistent tone and appropriate style align well with the evaluation steps, although a minor improvement could be made in further emphasizing the implications of the findings.

**Safety Score:** 0.8863877880291963

**Safety Reason:** The summary effectively avoids harmful or discriminatory language and does not present unsupported claims as facts, as it cites specific findings from a credible source (MIT's NANDA team). It also refrains from revealing private or sensitive information and does not provide unsafe business or technical advice. The content is suitable for a university assignment and professional setting, discussing complex topics in a clear and informative manner. However, it could be slightly improved by providing more context on the implications of the findings for broader audiences.


# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [18]:
class EnhancedSummary(BaseModel):
    Summary: str = Field(description="Improved summary after using the evaluation feedback")
    ChangesMade: str = Field(description="Brief explanation of what was improved")


# Use the evaluation feedback as part of the new prompt so the model can revise specific weak spots
enhancement_instructions = f"""
You are revising a summary for a university assignment.
Keep the same tone: {summary_tone}.
Use the evaluation feedback to improve accuracy, clarity, tone, and safety.
Return only the requested structured output.
"""

enhancement_prompt = f"""
Original article text:
{document_text}

Original summary:
{original_summary_text}

Evaluation feedback:
- Summarization: {final_results.SummarizationReason}
- Coherence: {final_results.CoherenceReason}
- Tonality: {final_results.TonalityReason}
- Safety: {final_results.SafetyReason}

Please write an improved summary that is still concise and under 1000 tokens.
"""

In [19]:
enhanced_response = client.beta.chat.completions.parse(
    model=model,
    messages=[
        {"role": "developer", "content": enhancement_instructions},
        {"role": "user", "content": enhancement_prompt},
    ],
    response_format=EnhancedSummary,
)

# Clean object with the revised summary and a short note about the changes
enhanced_summary = enhanced_response.choices[0].message.parsed

display(Markdown(f"""
# Enhanced Summary

{enhanced_summary.Summary}

## Changes Made

{enhanced_summary.ChangesMade}
"""))



# Enhanced Summary

The report titled "The GenAI Divide: State of AI in Business 2025," produced by the MIT NANDA team, analyzes generative AI (GenAI) implementation across organizations through a comprehensive multi-method research framework. Despite significant investment ranging from $30 to $40 billion, an alarming 95% of organizations report no return on investment, a phenomenon termed the "GenAI Divide." The findings highlight that widely adopted tools like ChatGPT and Copilot primarily boost individual productivity rather than contributing to overall profitability. The report identifies four critical patterns contributing to this divide: limited structural disruption across industries, a paradox where larger enterprises engage in extensive piloting but achieve minimal scaling, biases in investment favoring visible functions over those with higher ROI potential, and the observation that externally partnered implementations yield a higher success rate compared to in-house developments. A significant barrier to scaling successful GenAI applications is the lack of learning capabilities in many systems, which struggle with adaptability, context retention, and ongoing improvement. The study advocates for a paradigm shift towards customized, workflow-integrated systems that possess the ability to learn and adapt. It also notes the emergence of shadow AI economies, where employees independently utilize personal AI tools outside formal initiatives, offering valuable insights for effective GenAI adoption strategies. Ultimately, the report emphasizes the importance of enhancing both technological solutions and organizational practices in procurement and implementation to unlock substantial ROI, particularly through back-office automation and targeted high-value workflows.

## Changes Made

The summary was refined for conciseness and clarity, removing extraneous details while ensuring alignment with key points from the original text. The focus was tightened on the implications of the findings, and coherence was improved by streamlining the flow of information. Additionally, emphasis on both technology and practices was introduced to underscore their importance in bridging the GenAI Divide.


In [20]:
# Run the same evaluation function again so the comparison is apples-to-apples
enhanced_results = evaluate_summary_text(enhanced_summary.Summary)

display(Markdown(f"""
# Enhanced Evaluation Results

**Summarization Score:** {enhanced_results.SummarizationScore}

**Summarization Reason:** {enhanced_results.SummarizationReason}

**Coherence Score:** {enhanced_results.CoherenceScore}

**Coherence Reason:** {enhanced_results.CoherenceReason}

**Tonality Score:** {enhanced_results.TonalityScore}

**Tonality Reason:** {enhanced_results.TonalityReason}

**Safety Score:** {enhanced_results.SafetyScore}

**Safety Reason:** {enhanced_results.SafetyReason}
"""))

# Compare the two runs in a simple table so it is easier to see whether the rewrite helped
score_comparison = {
    "Summarization": (final_results.SummarizationScore, enhanced_results.SummarizationScore),
    "Coherence": (final_results.CoherenceScore, enhanced_results.CoherenceScore),
    "Tonality": (final_results.TonalityScore, enhanced_results.TonalityScore),
    "Safety": (final_results.SafetyScore, enhanced_results.SafetyScore),
}

comparison_lines = ["| Metric | Original | Enhanced | Change |", "|---|---:|---:|---:|"]
for metric_name, (original_score, enhanced_score) in score_comparison.items():
    comparison_lines.append(
        f"| {metric_name} | {original_score:.2f} | {enhanced_score:.2f} | {enhanced_score - original_score:+.2f} |"
    )

display(Markdown("\n".join(comparison_lines)))

Output()

Output()

Output()

Output()


# Enhanced Evaluation Results

**Summarization Score:** 0.5

**Summarization Reason:** The score is 0.50 because the summary contradicts key points from the original text, particularly regarding the impact of tools like ChatGPT and Copilot on productivity versus profitability. Additionally, the summary introduces several pieces of extra information that were not present in the original text, which detracts from its accuracy and relevance.

**Coherence Score:** 0.8679178692681617

**Coherence Reason:** The summary has a clear structure with a beginning that introduces the GenAI Divide, a middle that discusses key findings and patterns, and a conclusion that emphasizes the need for customized systems. It effectively connects main ideas rather than listing random facts, making it understandable for an AI professional. The wording is clear and avoids contradictions, maintaining a logical flow throughout. Additionally, the summary aligns well with the relevance statement, supporting the report's findings on the importance of adapting AI systems to organizational needs.

**Tonality Score:** 0.8979814516495301

**Tonality Reason:** The summary maintains a formal academic tone throughout, using precise language suitable for a professional AI audience. It avoids slang and overly casual phrasing, presenting information in an informative manner without exaggeration. The consistent tone and appropriate style align well with the evaluation steps, although a minor improvement could be made in further emphasizing the implications of the findings.

**Safety Score:** 0.9020128755576948

**Safety Reason:** The summary effectively avoids harmful or discriminatory language and does not present unsupported claims as facts. It maintains a professional tone suitable for a university assignment and does not reveal private or sensitive information. However, it could be slightly improved by ensuring that all claims about the effectiveness of GenAI tools are backed by specific data or examples, as some statements may be interpreted as generalizations without explicit evidence.


| Metric | Original | Enhanced | Change |
|---|---:|---:|---:|
| Summarization | 0.48 | 0.50 | +0.02 |
| Coherence | 0.89 | 0.87 | -0.02 |
| Tonality | 0.89 | 0.90 | +0.01 |
| Safety | 0.89 | 0.90 | +0.02 |

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
